In [1]:
import pickle
import numpy as np
import torch

In [2]:
# Load the NumPy array from a pickle file
with open('../Data/MP_data/displacements0-5000.pkl', 'rb') as file:
    dis_array = pickle.load(file)

In [3]:
dis_array.shape

(5000, 1317, 3)

In [4]:
from neuralop.layers.neighbor_search import NeighborSearch
from neuralop.layers.integral_transform import IntegralTransform

In [5]:
mesh = np.loadtxt('../Data/dataset/mesh.csv', delimiter=',')

In [6]:
print(np.max(mesh[0,:]), np.max(mesh[1,:]))

2.5 0.41


In [7]:
idx_x = torch.arange(0,2.5, 0.01)
idx_y = torch.arange(0,.41, 0.01)
x, y  = torch.meshgrid(idx_x, idx_y, indexing='ij') 


In [8]:
print(x.shape, y.shape, idx_x.shape, idx_y.shape)

torch.Size([250, 41]) torch.Size([250, 41]) torch.Size([250]) torch.Size([41])


In [9]:
simple_mesh = torch.stack([x.flatten(), y.flatten()]).type(torch.float)

In [10]:
com_mesh = torch.stack( [torch.tensor(mesh[0,:]),torch.tensor(mesh[1,:])]).type(torch.float)

In [11]:
print(com_mesh.shape, simple_mesh.shape)

torch.Size([2, 1317]) torch.Size([2, 10250])


In [21]:
NS = NeighborSearch(use_open3d=False)
pos_dict_tosimple = NS(torch.transpose(com_mesh, 0, 1), torch.transpose(simple_mesh,0, 1), radius=0.1)

In [22]:
temp = pos_dict_tosimple['neighbors_row_splits']
temp2 = temp[1:]-temp[:-1]
print(min(temp2),max(temp2))
print(len(temp2))

tensor(3) tensor(421)
10250


In [23]:
print(pos_dict_tosimple.keys())

dict_keys(['neighbors_index', 'neighbors_row_splits'])


In [24]:
NS_2 = NeighborSearch(use_open3d=False)
pos_dict_tocomplex = NS_2( torch.transpose(simple_mesh,0, 1),torch.transpose(com_mesh, 0, 1), radius=0.04)

In [25]:
temp = pos_dict_tocomplex['neighbors_row_splits']
temp2 = temp[1:]-temp[:-1]
print(min(temp2),max(temp2))
print(len(temp2))

tensor(8) tensor(52)
1317


In [26]:
it = IntegralTransform(mlp_layers=[4,1]).cuda()

In [27]:
f = com_mesh[0,:]+com_mesh[1,:]
print(f.shape)

torch.Size([1317])


In [29]:
for key, value in pos_dict_tosimple.items():
    pos_dict_tosimple[key] = pos_dict_tosimple[key].cuda()

f_x = it(torch.transpose(com_mesh, 0, 1).cuda(),pos_dict_tosimple, torch.transpose(simple_mesh,0, 1).cuda(),f.reshape(-1,1).cuda() )

In [30]:
f_x.shape

torch.Size([10250, 1])

In [50]:
torch.transpose(simple_mesh,0, 1).shape

torch.Size([10250, 2])